## AVH data resources

In [3]:
import pandas as pd
import requests

url = 'https://api.ala.org.au/occurrences/occurrences/facets'
params = {
  'q': 'data_hub_uid:dh9',
  'facets': 'data_resource_uid'
}

r = requests.get(url, params=params)

data = r.json()

data_resources = []
for row in data[0]['fieldResult']:
  data_resources.append({
    'uid': row['i18nCode'][row['i18nCode'].find('.') + 1:],
    'label': row['label'],
    'count': row['count']
  })

df_data_resources = pd.DataFrame(data_resources)

## Occurrences with annotations for data resource

In [4]:
def create_occurrences_df_for_dr(dr):
  url = 'https://api.ala.org.au/occurrences/occurrences/search'
  pageSize = 1000
  start = 0
  totalRecords = 1
  occurrences = []
  while start < totalRecords:
    params = {
      'q': f'data_resource_uid:{dr}',
      'fq': 'user_assertions:*',
      'fl': 'id,data_resource_uid,institutionCode,collectionCode,catalogNumber',
      'pageSize': pageSize,
      'start': start
    }
    r = requests.get(url, params=params)
    data = r.json()
    totalRecords = data['totalRecords']
    occurrences += data['occurrences']
    start += pageSize
  return pd.DataFrame(occurrences).rename(columns={
    'raw_institutionCode': 'institutionCode',
    'raw_collectionCode': 'collectionCode',
    'uuid': 'occurrenceID',
    'raw_catalogNumber': 'catalogNumber'
  })

## Annotations for data resource

In [5]:
def create_annotations_df_for_dr(dr):
  df_occurrences = create_occurrences_df_for_dr(dr)
  if df_occurrences.empty:
    return pd.DataFrame([])
  annotations = []
  for index, row in df_occurrences.iterrows():
    uid = row['occurrenceID']
    url = f'https://biocache.ala.org.au/ws/occurrences/{uid}/assertions'
    r = requests.get(url)
    data = r.json()
    annotations += sorted(data, key=lambda item: item['created'])
  df_annotations = pd.DataFrame(annotations)
  return pd.merge(df_occurrences, df_annotations, how="left", left_on="occurrenceID", right_on="referenceRowKey")

## Annotations for AVH DRs

In [6]:
for index, row in df_data_resources.iterrows():
  df = create_annotations_df_for_dr(row['uid'])
  if not df.empty:
    df.to_excel('data/' + row['uid'] + '.xlsx', index=False)

## Annotations for Museum DRs

(selected museums)

In [7]:
museum_drs = ['dr342', 'dr340', 'dr344']
for dr in museum_drs:
  df = create_annotations_df_for_dr(dr)
  if not df.empty:
    df.to_excel('data/' + dr + '.xlsx', index=False)

## Annotations for NRCA DRs

In [12]:
nrca_drs = ['dr341', 'dr349', 'dr130', 'dr15860', 'dr15867', 'dr15868']
for index, dr in enumerate(nrca_drs):
  if index == 0:
    df = create_annotations_df_for_dr(dr)
  else: 
    df = pd.concat([df, create_annotations_df_for_dr(dr)], ignore_index=True)

  df.rename(columns={
    'dataResourceUid_x': 'dataResourceUid',
  }).drop(columns=['dataResourceUid_y', 'userEmail']).to_excel('data/nrca.xlsx', index=False)